# WattMosaic

### Domain-aware campus energy forecasting under incomplete sensor data

**Originally developed for NTU Datathon / Deep Learning Week 2026 — Track 1: Smart Campus Analytics.**

WattMosaic predicts campus facility energy consumption from building context, calendar signals, environmental conditions, occupancy, and previous usage. The final approach focuses on a practical issue in real telemetry: **different sensors can be missing in different combinations**.

**Verified public leaderboard RMSE:** `3.17093`

> Public release note: the competition datasets, serialized model weights, and generated prediction files are intentionally not redistributed. This notebook documents the final modelling approach and reproducibility interface without publishing restricted or submission-only artifacts.


## 1. Problem framing

The target is continuous campus-facility energy consumption. Useful predictors include building identity/type, hour, weekday, month, temperature, humidity, occupancy, and previous usage.

A major practical challenge was **distribution shift in sensor missingness**: the reference/test features contained different missing-sensor combinations from the labelled training data. That motivated a mask-aware model rather than a single generic imputation path.


In [ ]:
SENSORS = ['temperature', 'humidity', 'occupancy', 'previous_usage']

def missingness_mask(frame):
    """Encode the four sensor availability states as an integer 0..15."""
    import numpy as np
    return frame[SENSORS].isna().to_numpy(dtype=np.int64) @ (2 ** np.arange(len(SENSORS)))

# Four binary sensor states => 16 routing patterns.
assert 2 ** len(SENSORS) == 16


## 2. Final architecture

The final system uses a shared energy core plus **15 specialists**, one for each non-empty missing-sensor mask. Complete rows use the core directly; incomplete rows are routed to the specialist corresponding to the sensors that are unavailable.

The specialists are **domain-aware**: unlabeled reference features inform the relationship among observable sensors, but hidden energy labels are never used. This improved transfer to the hackathon leaderboard compared with generic imputation and several larger ensembles.


In [ ]:
def route_prediction(core_predict, specialist_predict, frame):
    """Simplified version of WattMosaic's mask-aware inference routing."""
    import numpy as np
    masks = missingness_mask(frame)
    out = np.empty(len(frame), dtype=float)

    for mask in np.unique(masks):
        idx = np.flatnonzero(masks == mask)
        subset = frame.iloc[idx]
        base = core_predict(subset, int(mask))
        if mask == 0:
            out[idx] = base
        else:
            out[idx] = base + specialist_predict(subset, int(mask))

    return np.maximum(out, 0.0)


## 3. Validation strategy

Model selection did not rely on a single shuffled cross-validation score. The development process compared ordinary missingness, **test-like joint sensor dropout**, heavier dropout stress tests, and public leaderboard feedback from materially different candidate models.

The central lesson was that a model with slightly better ordinary CV could still transfer worse when sensor availability shifted. Validation therefore emphasized the deployment-like missingness regime.


In [ ]:
import pandas as pd

results = pd.DataFrame([
    {'evaluation': 'Development audit — natural missingness', 'RMSE': 3.022861, 'MAE': 2.381042, 'R2': 0.973150},
    {'evaluation': 'Development audit — test-like joint missingness', 'RMSE': 3.214310, 'MAE': 2.507000, 'R2': 0.969600},
    {'evaluation': 'Hackathon public leaderboard', 'RMSE': 3.170930, 'MAE': None, 'R2': None},
])
results


## 4. Runtime engineering

The final competition environment was fixed to **Python 3.12** and specific package versions. Because the selected specialists had originally been trained with CatBoost while CatBoost was not part of the final allowed dependency list, the fitted tree ensembles were exported to a portable representation containing plain Python model data.

This decoupled inference from CatBoost at runtime while preserving the selected predictions numerically. The final submission also validated output schema, ID order, finite values, batching behavior, unseen IDs, and every one of the 16 missingness patterns.


In [ ]:
EXPECTED_RUNTIME = {
    'python': '3.12',
    'joblib': '1.4.2',
    'lightgbm': '4.5.0',
    'matplotlib': '3.9.2',
    'numpy': '2.1.3',
    'pandas': '2.2.3',
    'scikit-learn': '1.5.2',
    'scipy': '1.14.1',
    'seaborn': '0.13.2',
    'xgboost': '2.1.3',
}
EXPECTED_RUNTIME


## 5. Public-release boundaries

This notebook is intentionally a **portfolio-oriented technical walkthrough**, not the raw competition workspace.

Excluded from the public repository:

- organizer-provided training/test datasets and submission template;
- serialized final model weights;
- generated leaderboard prediction CSVs;
- intermediate Round 1–7 experiments and probing artifacts;
- internal validation dumps and submission checklists.

The original materials did not explicitly grant public redistribution rights for the supplied datasets, so those files are not republished here.


## 6. Takeaways

1. **Missingness is part of the prediction problem.** Different sensor failures can warrant different experts.
2. **Better imputation is not automatically better downstream prediction.** Optimize the task metric.
3. **Validation should resemble deployment.** Test-like missingness checks were more informative than generic CV alone.
4. **Complexity must earn its place.** Several larger boosting and ensemble variants were rejected when they failed to transfer.
5. **Reproducibility is a modelling feature.** Runtime pinning, portable inference, and defensive output checks mattered under competition constraints.
